# 01 - Cognitive Evaluation Prompt Tuning

Validate that the LLM can reliably produce structured JSON with precise importance floats and emotion enum labels from conversational context.

- **Toolchain**: `instructor` + `pydantic` for typed output enforcement
- **Goal**: Iterate on `EngramScore` field descriptions until the output distribution aligns with memory decay threshold expectations

In [ ]:
from dotenv import load_dotenv
import os
import instructor
from pydantic import BaseModel, Field
from enum import Enum
from openai import OpenAI

load_dotenv("../.env")

MODEL = os.environ.get("LLM_MODEL", "gpt-5.1")

client = instructor.from_openai(
    OpenAI(base_url=os.environ["LLM_API_URL"], api_key=os.environ["LLM_API_KEY"]),
    mode=instructor.Mode.JSON,
)

In [ ]:
class Emotion(str, Enum):
    # Engineering baseline
    NEUTRAL = "neutral"
    # Plutchik 8 primaries (4 polar pairs)
    JOY = "joy"
    SADNESS = "sadness"
    TRUST = "trust"
    DISGUST = "disgust"
    FEAR = "fear"
    ANGER = "anger"
    SURPRISE = "surprise"
    ANTICIPATION = "anticipation"
    # Ekman extension
    CONTEMPT = "contempt"

class EmotionWeight(BaseModel):
    emotion: Emotion
    weight: float = Field(
        ge=0.0, le=1.0,
        description="Contribution weight of this emotion (0.0-1.0)."
    )

class EngramScore(BaseModel):
    emotions: list[EmotionWeight] = Field(
        min_length=1, max_length=3,
        description=(
            "Emotional composition of the conversation. "
            "1-3 emotions ranked by dominance. Weights must sum to 1.0. "
            "Use 1 emotion for simple cases, 2-3 for mixed/complex cases."
        ),
    )
    importance: float = Field(
        ge=0.0, le=1.0,
        description=(
            "Memory importance weight (0.0-1.0). "
            "Output exactly 2 decimal places. "
            "Avoid multiples of 0.05 (e.g. 0.10, 0.25, 0.80) unless truly warranted. "
            "Prefer granular values like 0.07, 0.13, 0.42, 0.68, 0.91, 0.97.\n"
            "0.00-0.19: Meaningless small talk, greetings, filler.\n"
            "0.20-0.49: Ordinary factual exchange, general Q&A.\n"
            "0.50-0.79: Personal preferences, health constraints, habits, hobbies, "
            "daily concerns, emotional venting, in-depth sharing.\n"
            "0.80-1.00: Identity-defining core memory ONLY — real name, major life "
            "events (death, birth, marriage), long-term mental health diagnosis, "
            "fundamental relationship changes. Must permanently alter the persona's "
            "world model."
        ),
    )
    reasoning: str = Field(
        description="Brief justification for the weight and emotion assignment, max 20 words."
    )

In [ ]:
SYSTEM_PROMPT = (
    "You are the cognitive scorer of the Pgramma engine. "
    "Objectively evaluate the memory-retention value of the following conversation. "
    "Respond ONLY with the structured JSON schema provided."
)

def evaluate_memory(chat_history: str) -> EngramScore:
    """Score a conversation snippet for emotion and memory importance."""
    return client.chat.completions.create(
        model=MODEL,
        response_model=EngramScore,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": chat_history},
        ],
    )

## Batch Stability Test

Each case runs `N_RUNS` times. We collect importance scores to measure mean, std, and visualize distribution.

In [ ]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

N_RUNS = 5

test_cases = [
    # --- Trivial (expected 0.00-0.19) ---
    ("User: Hi\nAssistant: Hey!", "trivial-greeting"),
    ("User: lol\nAssistant: Haha 😄", "trivial-filler"),
    ("User: Okay thanks\nAssistant: No problem!", "trivial-closing"),

    # --- Factual (expected 0.20-0.49) ---
    ("User: What's the capital of France?\nAssistant: Paris.", "fact-geography"),
    ("User: How many bytes in a kilobyte?\nAssistant: 1024.", "fact-tech"),
    ("User: When did WW2 end?\nAssistant: 1945.", "fact-history"),

    # --- Personal (expected 0.50-0.79) ---
    ("User: I'm severely allergic to cat fur, never show me cat pics.\n"
     "Assistant: Noted, I'll avoid anything cat-related.", "personal-allergy"),
    ("User: I've been feeling really burnt out at work lately.\n"
     "Assistant: That sounds exhausting. Want to talk about it?", "personal-burnout"),
    ("User: My favorite band is Radiohead, I've seen them live 3 times.\n"
     "Assistant: That's awesome, Radiohead is legendary.", "personal-hobby"),
    ("User: I'm learning Japanese, it's been my dream since childhood.\n"
     "Assistant: That's a beautiful goal. How far along are you?", "personal-goal"),
    ("User: I just can't stand people who litter, it drives me insane.\n"
     "Assistant: Understandable, it shows a lack of basic respect.", "personal-anger"),
    ("User: I absolutely hate the taste of cilantro, it ruins any dish for me.\n"
     "Assistant: Noted, I'll keep that in mind for any food suggestions.", "personal-disgust"),
    ("User: Sometimes I miss the summers at my grandma's countryside house.\n"
     "Assistant: Those sound like precious memories.", "personal-nostalgia"),
    ("User: My exam results come out tomorrow, I can barely sleep.\n"
     "Assistant: The waiting is always the hardest part. Good luck!", "personal-anticipation"),
    ("User: My ex keeps pretending we're still friends. It's so fake and pathetic.\n"
     "Assistant: That sounds really frustrating to deal with.", "personal-contempt"),

    # --- Core (expected 0.80-1.00) ---
    ("User: My name is Kira. I just lost my mother yesterday.\n"
     "Assistant: I'm so sorry for your loss, Kira.", "core-bereavement"),
    ("User: I've decided to cut off my father permanently. He's been abusive my whole life.\n"
     "Assistant: That must have taken immense courage.", "core-relationship"),
    ("User: I was diagnosed with depression three years ago.\n"
     "Assistant: Thank you for sharing that with me.", "core-diagnosis"),
    ("User: I proposed to my girlfriend today and she said yes!\n"
     "Assistant: Congratulations! That's wonderful news!", "core-milestone"),
    ("User: I just found out I'm pregnant, I can't believe it!\n"
     "Assistant: Oh wow, that's life-changing news!", "core-surprise"),
]

# Parallel evaluation
results = defaultdict(lambda: {"scores": [], "emotions_list": []})

def run_single(chat, label, run_idx):
    score = evaluate_memory(chat)
    return label, run_idx, score

futures = []
with ThreadPoolExecutor(max_workers=8) as executor:
    for run in range(N_RUNS):
        for chat, label in test_cases:
            futures.append(executor.submit(run_single, chat, label, run))

    for future in as_completed(futures):
        label, run_idx, score = future.result()
        results[label]["scores"].append(score.importance)
        emo_str = " + ".join(f"{e.emotion.value}({e.weight:.1f})" for e in score.emotions)
        results[label]["emotions_list"].append(emo_str)

# Print summary per label (ordered by test_cases)
seen = []
for _, label in test_cases:
    if label not in seen:
        seen.append(label)
for label in seen:
    scores = results[label]["scores"]
    emos = results[label]["emotions_list"]
    print(f"[{label:<24}]  imp={[f'{s:.2f}' for s in scores]}")
    for emo in emos:
        print(f"  {' ' * 26}{emo}")

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Ordered labels following test_cases
seen = []
for _, tag in test_cases:
    if tag not in seen:
        seen.append(tag)
labels = seen

means = [np.mean(results[tag]["scores"]) for tag in labels]
stds = [np.std(results[tag]["scores"]) for tag in labels]

# --- Fig 1: Importance mean ± std bar chart ---
colors = []
for tag in labels:
    if tag.startswith("trivial"):
        colors.append("#9e9e9e")
    elif tag.startswith("fact"):
        colors.append("#42a5f5")
    elif tag.startswith("personal"):
        colors.append("#ffa726")
    else:
        colors.append("#ef5350")

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=labels, y=means,
    error_y=dict(type="data", array=stds, visible=True),
    marker_color=colors, marker_line=dict(color="black", width=0.5),
    showlegend=False,
))

# Zone shading
for y0, y1, color in [
    (0.0, 0.19, "gray"),
    (0.2, 0.49, "blue"),
    (0.5, 0.79, "orange"),
    (0.8, 1.0, "red"),
]:
    fig1.add_hrect(y0=y0, y1=y1, fillcolor=color, opacity=0.08, line_width=0)

fig1.update_layout(
    title=f"Importance Score: Mean ± Std  (N={N_RUNS} runs per case)",
    yaxis_title="Importance", yaxis_range=[0, 1.05],
    xaxis_tickangle=-45, height=450, template="plotly_white",
)
fig1.show()

# --- Fig 2: Weighted emotion heatmap ---
all_emotions = [e.value for e in Emotion]
emotion_matrix = np.zeros((len(labels), len(all_emotions)))

for col, tag in enumerate(labels):
    for emo_str in results[tag]["emotions_list"]:
        for part in emo_str.split(" + "):
            name, w = part.rsplit("(", 1)
            row = all_emotions.index(name)
            emotion_matrix[col][row] += float(w.rstrip(")"))

# Normalize per case: divide by N_RUNS to get average weight
emotion_matrix /= N_RUNS

z = emotion_matrix.T  # shape: (n_emotions, n_labels)
text = [[f"{z[r][c]:.2f}" if z[r][c] > 0.05 else ""
         for c in range(z.shape[1])] for r in range(z.shape[0])]

fig2 = go.Figure()
fig2.add_trace(go.Heatmap(
    z=z, x=labels, y=all_emotions,
    colorscale="YlOrRd", zmin=0, zmax=1,
    text=text, texttemplate="%{text}",
    colorbar=dict(title="Avg Weight"),
    hovertemplate="case: %{x}<br>emotion: %{y}<br>weight: %{z:.2f}<extra></extra>",
))
fig2.update_layout(
    title=f"Emotion Weight Distribution (avg over N={N_RUNS} runs)",
    xaxis_tickangle=-45, height=450, template="plotly_white",
)
fig2.show()

# --- Stats summary ---
print(f"{'Label':<24} {'Mean':>6} {'Std':>6} {'Min':>6} {'Max':>6}  Dominant Emotions (avg weight)")
print("-" * 90)
for tag in labels:
    scores = results[tag]["scores"]
    col = labels.index(tag)
    top_indices = np.argsort(emotion_matrix[col])[::-1]
    top_str = ", ".join(
        f"{all_emotions[i]}={emotion_matrix[col][i]:.2f}"
        for i in top_indices if emotion_matrix[col][i] > 0.05
    )
    print(f"{tag:<24} {np.mean(scores):>6.2f} {np.std(scores):>6.3f} {min(scores):>6.2f} {max(scores):>6.2f}  {top_str}")